# 04_gov_<dataset>_<table> — table-scoped governance review

This notebook selects a logical table from `METADATA_DATA_CATALOGUE` and records approved governance metadata only after explicit human commit actions. It does not require a Data Agreement and does not enforce DQ or classification rules in `03_pc`.

## 1. Run `00_env_config`

In [ ]:
%run 00_env_config


## 2. Import supported public APIs

In [ ]:
from fabricops_kit import (
    build_classification_records,
    build_column_context_records,
    build_dq_rule_records,
    build_profile_summary,
    commit_column_classification,
    commit_column_context,
    commit_dq_rules,
    get_selected_catalogue_table,
    latest_by_column,
    load_catalogue_profile_rows,
    optional_ai_generate_response,
    read_lakehouse_table,
    widget_review_table_governance,
    widget_select_catalogue_table,
)


## 3. Select catalogue table

In [ ]:
env_name = ENV_NAME

# Reads METADATA_DATA_CATALOGUE from the configured metadata target and selects
# one logical table using the latest successful profile run.
widget_select_catalogue_table(CONFIG, env_name, spark_session=spark)
selected_table = get_selected_catalogue_table()
selected_table


## 4. Show selected profile summary

In [ ]:
profile_rows = load_catalogue_profile_rows(CONFIG, env_name, selected_table, spark_session=spark)
profile_summary = build_profile_summary(profile_rows, selected_table)
display(spark.createDataFrame([profile_summary]))
display(spark.createDataFrame(profile_rows))


## 5. Review and commit business context

The review state below is intentionally non-persistent. Edit the `reviewed_context_rows` list, set `commit=True` only for approved rows, then run the commit cell. AI suggestions may be copied into `ai_suggestion_json`, accepted, ignored, or changed before commit.

In [ ]:
try:
    existing_context_rows = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_COLUMN_CONTEXT", spark_session=spark)
    existing_context = latest_by_column(existing_context_rows)
except Exception:
    existing_context = {}

review_state = widget_review_table_governance(profile_rows, existing_context=existing_context)

# Optional Fabric AI example. If Fabric AI is unavailable, this returns None and manual review continues.
# prepared_context_df = spark.createDataFrame(profile_rows)
# ai_context_df = optional_ai_generate_response(
#     prepared_context_df,
#     prompt="Suggest concise business context for {table_name}.{column_name} using only profile metrics.",
#     output_col="ai_business_context_suggestion",
# )

reviewed_context_rows = [
    # {"column_name": "example_column", "business_context": "Human-approved meaning", "notes": "Reviewed in 04_gov", "review_status": "approved", "commit": True}
]
context_records = build_column_context_records(profile_rows, reviewed_context_rows, config=CONFIG, env=env_name)
# Explicit human commit action. Nothing is written until this function is called.
commit_column_context(CONFIG, env_name, context_records, spark_session=spark)
print(f"Committed {len(context_records)} approved business-context row(s).")


## 6. Review and commit DQ rules

Author rules manually or use AI suggestions as advisory drafts. `03_pc` does not execute these approved rules in v1.0.0.

In [ ]:
try:
    existing_rule_rows = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_DQ_RULES", spark_session=spark)
    existing_rules = latest_by_column(existing_rule_rows)
except Exception:
    existing_rules = {}

# Optional AI suggestion example; unavailable AI does not block manual rule entry.
# ai_rule_df = optional_ai_generate_response(
#     spark.createDataFrame(profile_rows),
#     prompt="Suggest candidate DQ rules for {table_name}.{column_name}; return JSON only.",
#     output_col="ai_dq_rule_suggestion",
# )

reviewed_dq_rules = [
    # {"rule_id": "orders.order_id.not_null", "column_name": "order_id", "rule_type": "not_null", "rule_parameters": {}, "severity": "error", "description": "Human-approved not-null expectation", "review_status": "approved", "is_active": True, "commit": True}
]
dq_records = build_dq_rule_records(profile_rows, reviewed_dq_rules, config=CONFIG, env=env_name)
# Explicit human commit action. No rule is enforced or persisted before this call.
commit_dq_rules(CONFIG, env_name, dq_records, spark_session=spark)
print(f"Committed {len(dq_records)} approved DQ-rule row(s).")


## 7. Review and commit sensitivity and PII classification

In [ ]:
try:
    existing_classification_rows = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_COLUMN_CLASSIFICATION", spark_session=spark)
    existing_classification = latest_by_column(existing_classification_rows)
except Exception:
    existing_classification = {}

# Optional AI suggestion example. Human approval remains mandatory.
# ai_classification_df = optional_ai_generate_response(
#     spark.createDataFrame(profile_rows),
#     prompt="Suggest sensitivity_label and personal_data_classification for {table_name}.{column_name}; return JSON only.",
#     output_col="ai_classification_suggestion",
# )

reviewed_classification_rows = [
    # {"column_name": "customer_id", "sensitivity_label": "confidential", "personal_data_classification": "indirect_identifier", "pii_identifier_type": "customer key", "handling_requirement": "Limit to approved business users", "reasoning": "Human-reviewed identifier column", "review_status": "approved", "commit": True}
]
classification_records = build_classification_records(profile_rows, reviewed_classification_rows, config=CONFIG, env=env_name)
# Explicit human commit action. AI classification never counts as approval.
commit_column_classification(CONFIG, env_name, classification_records, spark_session=spark)
print(f"Committed {len(classification_records)} approved classification row(s).")


## 8. Display governance completion summary

In [ ]:
completion_summary = {
    "environment_name": selected_table["environment_name"],
    "dataset_name": selected_table["dataset_name"],
    "table_name": selected_table["table_name"],
    "profile_run_id": selected_table["profile_run_id"],
    "business_context_rows_committed": len(context_records),
    "dq_rule_rows_committed": len(dq_records),
    "classification_rows_committed": len(classification_records),
    "enforcement_scope": "Out of scope for v1.0.0; 03_pc keeps notebook-defined schema and drift guardrails only.",
}
display(spark.createDataFrame([completion_summary]))


## Notebook-output examples for PR review

- **Business context stage:** selected profile columns display with existing approved context and manual edit instructions.
- **DQ stage:** approved rules are append-only metadata events and are not executed by this notebook or `03_pc`.
- **Classification stage:** sensitivity and personal-data labels require human commit; AI is optional and advisory.